In [ ]:

# ============================================================
# EXPERIMENT 12
# Cloud-Based Generative AI Application
# Using Groq API and Gradio
# ============================================================

# 1. Install dependencies

!pip install -U groq gradio -q


# 2. Import required libraries

import os
import re
import time
import getpass
import gradio as gr
from groq import Groq


# ============================================================
# 3. Get Groq API Key
# ============================================================

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    api_key = getpass.getpass("Enter your Groq API key: ")

client = Groq(api_key=api_key)


# ============================================================
# 4. Find an available Groq model
# ============================================================

preferred_models = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b"
]

try:
    available_models = client.models.list()

    available_model_ids = [
        model.id for model in available_models.data
    ]

    selected_model = None

    for model_name in preferred_models:
        if model_name in available_model_ids:
            selected_model = model_name
            break

    if selected_model is None:
        raise Exception(
            "No supported text-generation model is available "
            "for this Groq API key."
        )

    print("Selected Model:", selected_model)

except Exception as error:
    print("Unable to retrieve available models.")
    print("Error:", error)
    selected_model = "llama-3.3-70b-versatile"


# ============================================================
# 5. Calculate Keyword Relevance Score
# ============================================================

def calculate_relevance(prompt, response):
    """
    Calculates a simple keyword-overlap relevance score.

    This is only a basic laboratory evaluation metric
    and is not a replacement for human or benchmark evaluation.
    """

    stop_words = {
        "the", "a", "an", "is", "are", "was", "were",
        "to", "of", "in", "on", "for", "and", "or",
        "with", "what", "how", "why", "write", "explain",
        "describe", "give", "about"
    }

    prompt_words = set(
        re.findall(
            r"\b[a-zA-Z]{3,}\b",
            prompt.lower()
        )
    )

    response_words = set(
        re.findall(
            r"\b[a-zA-Z]{3,}\b",
            response.lower()
        )
    )

    important_words = prompt_words - stop_words

    if not important_words:
        return 100.0

    matched_words = (
        important_words.intersection(response_words)
    )

    score = (
        len(matched_words)
        / len(important_words)
    ) * 100

    return round(score, 2)


# ============================================================
# 6. Generate and Evaluate Response
# ============================================================

def generate_and_evaluate(
    prompt,
    temperature,
    max_tokens
):
    """
    Sends the prompt to a cloud-hosted language model
    and returns the generated response with evaluation metrics.
    """

    if not prompt or not prompt.strip():
        return (
            "Please enter a valid prompt.",
            {
                "Status": "No prompt provided"
            }
        )

    try:

        # Start timer
        start_time = time.perf_counter()

        # Send request to Groq
        completion = client.chat.completions.create(

            model=selected_model,

            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a helpful Generative AI assistant. "
                        "Provide accurate, clear and well-structured answers."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=float(temperature),

            max_completion_tokens=int(max_tokens)
        )

        # End timer
        end_time = time.perf_counter()

        # Extract generated response
        generated_response = (
            completion
            .choices[0]
            .message
            .content
            .strip()
        )

        # Calculate response time
        latency = end_time - start_time

        # Calculate word count
        word_count = len(
            generated_response.split()
        )

        # Calculate character count
        character_count = len(
            generated_response
        )

        # Calculate relevance
        relevance_score = calculate_relevance(
            prompt,
            generated_response
        )

        # Evaluation metrics
        evaluation = {
            "Model": selected_model,

            "Response Time (seconds)": round(
                latency,
                3
            ),

            "Generated Word Count": word_count,

            "Generated Character Count": character_count,

            "Keyword Relevance Score (%)":
                relevance_score,

            "Temperature":
                float(temperature),

            "Maximum Tokens":
                int(max_tokens),

            "Status":
                "Successfully generated"
        }

        return (
            generated_response,
            evaluation
        )

    except Exception as error:

        return (
            "The application could not generate a response.",

            {
                "Status": "Error",
                "Error Message": str(error)
            }
        )


# ============================================================
# 7. Create Gradio Application
# ============================================================

with gr.Blocks() as application:

    gr.Markdown(
        """
        # Cloud-Based Generative AI Application

        Enter a prompt to generate content and evaluate
        the response produced by a cloud-hosted language model.
        """
    )

    # --------------------------------------------------------
    # Input and Output Sections
    # --------------------------------------------------------

    with gr.Row():

        # ====================================================
        # INPUT SECTION
        # ====================================================

        with gr.Column():

            prompt_input = gr.Textbox(
                label="Enter Prompt",

                placeholder=(
                    "Example: Explain the applications "
                    "of Generative AI in education."
                ),

                lines=6
            )

            temperature_input = gr.Slider(
                minimum=0.0,
                maximum=1.0,
                value=0.3,
                step=0.1,

                label="Temperature"
            )

            max_tokens_input = gr.Slider(
                minimum=50,
                maximum=1000,
                value=250,
                step=50,

                label="Maximum Tokens"
            )

            generate_button = gr.Button(
                "Generate and Evaluate"
            )

            clear_button = gr.ClearButton(
                components=[
                    prompt_input,
                    response_output if "response_output" in globals()
                    else prompt_input
                ]
            )

        # ====================================================
        # OUTPUT SECTION
        # ====================================================

        with gr.Column():

            response_output = gr.Textbox(
                label="Generated Response",
                lines=14
            )

            evaluation_output = gr.JSON(
                label="Evaluation Metrics"
            )


    # ========================================================
    # 8. Connect Button
    # ========================================================

    generate_button.click(
        fn=generate_and_evaluate,

        inputs=[
            prompt_input,
            temperature_input,
            max_tokens_input
        ],

        outputs=[
            response_output,
            evaluation_output
        ]
    )


# ============================================================
# 9. Launch Application
# ============================================================

application.launch(
    share=True,
    debug=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 44.2 MB/s eta 0:00:00
Enter your Groq API key: ··········
Selected Model: openai/gpt-oss-20b
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1bb285c7d79a5a51a1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
